# The Annotated Transformer

References
- The annotated transformer blogpost by Harvard NLP: https://nlp.seas.harvard.edu/annotated-transformer/
- Attention is All You Need
- Layer Normalization

## The Paper

### Overview

The paper specifically investigates the sequence transduction models which typically follow an encoder-decoder architecture based on complex recurrent or convolutional neural networks. The paper proposes a (later-found-to-be) revolutionary network architecture called the **Transformer**, which based solely on attention mechanisms and delivers SOTA performance on machine translation tasks and also shows to generalize well to other tasks.

### The Original Goal

Sequence modeling and transduction problems such as language modeling and machine translation, which typically rely on recurrent models and encoder-decoder architectures to achieve SOTA results.

### What Are Recurrent Models?

Recurrent models generate a sequence of hidden states $h_t$ as a function of the previous hidden state $h_{t - 1}$ and the input for position $t$, which has an inherently sequential nature that makes it hard for parallelization and suffers memory issues from this fundamental constraint of sequential computation.

### How Does Attention Help?

The attention mechanism has been providing concrete improvement in sequence modeling and transduction models as they allow modeling of dependencies without regard to their distance in the input or output sequences.

### The Architecture

The transformer model proposed also follows a basic encoder-decoder architecture. Specifically, the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$. Then given the representations $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time in a *autoregressive* manner, which consumes the previously generated symbols as an additional input while generating the next. The transformer follows this architecture by using *stacked* self-attention and pointwise, fully connected layers for both the encoder and the decoder, as shown in the diagram below.
The transformer model proposed also follows a basic encoder-decoder architecture. Specifically, the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$. Then given the representations $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time in a *autoregressive* manner, which consumes the previously generated symbols as an additional input while generating the next. The transformer follows this architecture by using *stacked* self-attention and pointwise, fully connected layers for both the encoder and the decoder, as shown in the diagram below.

![Transformer architecture](imgs/transformer-arch.png)

#### The Encoder and the Decoder Stacks

##### Encoder

The encoder network is composed of a stack of $N = 67$ identical layers. Each layers has two sublayers:
- The first layer is a Multi-Head Self-Attention Mechanism
- The second layer is a simple, positionwise fully connected feed-forward network.
The encoder network also incorporates a residual connection around each two sublayers, followed by layer normalization, which means that the output of each sublayer is in the format,
$$\mathtt{LayerNorm}(x + \mathtt{SubLayer}(x)),$$
where $\mathtt{SubLayer}(x)$ represents the sublayer function iteself. To facilitate these residual connections, we keep all sublayers in the model, as well as the embedding layers produce outputs of the same dimension $d_\texttt{model} = 512$.

##### Decoder

The decoder network is also composed of a stack of $N = 6$ identical layers. In addition to the two sublayers in each encoder layer, the decoder adds in a third sublayer, which performs Multi-Head Attention over the output of the encoder stack. We also incorporate residual connections around each of the sublayers followed by the layer normalization as in the encoder network. Furthermore, we modify the self-attention sublayer in the decoder stack with causality masking to prevent attending to future tokens during generation, ensure that the predictions for position $i$ depend only on the previously known outputs at positions before $i$.

#### What is Layer Normalization?

Normalization serves as a technique to modify the computations performed during training of the deep neural networks based on stochastic gradient descent algorithms to make the learning easier. Batch normalization standardizes each summed input using its mean and standard deviation across the training data, which helps feed forward neural networks converge faster even with simple SGD. In addition, the stochasticity from the batch statistics serves as a regularizer during training.

However, batch normalization requires running averages of the summed input statistics, which can be computed easily in feed forward networks as it is easy to store statistics separately for each hidden layer, but can be tricky for recurrent neural networks as the recurrent neurons often require different statistics at different timesteps due to the varying sequence length. Batch normalization also cannot be applied to online learning tasks or to extremely large distributed models where the minibatches have to be small.

##### Background: The Batch Normalization

![Batch Normalization](imgs/batch-norm.png)

A feed forward neural network is a nonlinear mapping from an input pattern $\boldsymbol{x}$ to an output vector $\boldsymbol{y}$. Consider the $l^{th}$ hidden layer, and let $a^l$ be the vector representations of the summed inputs to the neurons in that layer, we have the summed inputs are computed via a linear projection with the weight matrix $W^l$ and the bottom-up inputs $h^l$ (hidden state) as follows,
$$a^l_i = w_i^{l \top} h^l, h_i^{l + 1} = f(a_i^l + b_i^l),$$
where $f(\cdot)$ is an elementwise nonlinear function and $w_i^l$ is the incoming weights to the $i^{th}$ hidden units and $b_i^l$ is the scalar bias parameter. The parameters of the neural network can be learned using modern SGD based algorithms. Batch normalization normalizes the summed inputs to each hidden unit over the training cases. Specifically, for the $i^{th}$ summed input in the $l^{th}$ layer, the batch normalization method rescales the summed inputs according to their variances under the distribution of the data
$$\bar{a}_i^l = \frac{g_i^l}{\sigma_i^l} (a_i^l - \mu_i^l), \quad \text{where } \mu_i^l = \underset{\boldsymbol{x} \sim p(\boldsymbol{x})}{\mathbb{E}} [a_i^l],\: \sigma_i^l = \sqrt{\underset{\boldsymbol{x} \sim p(\boldsymbol{x})}{\mathbb{E}} \left[ (a_i^l - \mu_i^l)^2 \right]},$$
where $\bar{a}_i^l$ is normalized summed inputs to the $i^{th}$ hidden unit in the $l^{th}$ layer and $g_i$ is the gain parameter scaling the normalized activation before the nonlinear activation function. Here the expectation is under the whole training data distribution. Thus it is usually impractical to compute the expectations in the above equations exactly, since it would require forward passes through the whole training dataset with the current set of weights (one pass). Hence we estimate $\mu$ and $\sigma$ using the empirical samples from the current minibatch, which puts constraints on the size of the minibatch and hard to apply to recurrent neural networks.

##### The Layer Normalization

![Layer Normalization](imgs/layer-norm.png)

Now let's consider the layer normalization designed to overcome the issues of batch normalization. Since the changes in the output of one layer will tend to cause highly correlated changes in the summed inputs to the next layer, especially with ReLU like units whose outputs can change by a lot, we can consider smoothing out the loss landscape by fixing the mean and the variance of the summed inputs within each layer. Thus, we can compute the layer normalization statistics over all the hidden units in the same layer as follows,
$$\mu^l = \frac{1}{H} \sum_{i = 1}^H a_i^l,\quad \sigma^l = \sqrt{\frac{1}{H} \sum_{i = 1}^H (a_i^l - \mu^l)^2},$$
where $H$ denotes the number of hidden units in a layer. The key difference between layer normalization and batch normalization is that under layer normalization, all the hidden units in a layer share the same normalization terms $\mu$ and $\sigma$, and unlike batch normalization, the layer normalization does not impose any size constraint of the minibatch and thus can work in online learning settings with even batch size $1$.

For recurrent neural networks, where the summed inputs in the recurrent layer are computed from the curren tinput $\boldsymbol{x}^t$ and the previous vector of hidden states $\boldsymbol{h}^{t - 1}$, which are computed as,
$$\boldsymbol{a}^t = W_{hh} h^{t - 1} + W_{xh} \boldsymbol{x}^t.$$
The layer normalized recurrent layer recenters and rescales its activations as follows,
$$\boldsymbol{h}^t = f\left[ \frac{\boldsymbol{g}}{\sigma^t} \odot (\boldsymbol{a}^t - \mu^t) + \boldsymbol{b} \right],\quad \text{where } \mu^t = \frac{1}{H} \sum_{i = 1}^{H} a_i^t,\: \sigma^t = \sqrt{\frac{1}{H} \sum_{i = 1}^{H} (a_i^t - \mu^t)^2}.$$

Note that in a standard RNN, there is a tendency for the average magnitude of the summed inputs to the recurrent units to either grow or shrink at every timesetp, leading to exploding or vanishing gradients. While using layer normalizations, however, the normalization terms make it invariant to rescaling all of the summed inputs to a layer, which results in a much more stable hidden-to-hidden dynamics during training.

##### `PyTorch` Implementation

In PyTorch, the `torch.nn.LayerNorm` class implements the operation,
$$y = \frac{x - \mathbb{E} [x]}{\sqrt{\mathrm{Var}[x] + \epsilon}} \ast \gamma + \beta,$$
where $\gamma$ and $\beta$ are learnable affine transform parameters when activated.

## The Code

In [3]:
# required imports
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import DataLoader
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

warnings.filterwarnings("ignore")
RUN_EXAMPLES = True # set to False to skip example runs

### The Model Architecture

Most SOTA neural sequence transduction models have an encoder-decoder architecture, where the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$,
$$(x_1, \cdots, x_n) \quad \xmapsto{\scriptsize\mathtt{Encoder}} \quad (z_1, \cdots, z_n).$$
Then given $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time. At each step the model is autoregressive, consuimg the previously generated symbols as additional input while generating the next symbol.

In [4]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture for sequence modeling and transduction.
    """
    def __init__(self, encoder, decoder, source_embedding, target_embedding, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.source_embedding = source_embedding
        self.target_embedding = target_embedding
        self.generator = generator
    
    def forward(self, source, target, source_mask, target_mask):
        """
        Forward pass to process the masked source and target sequences.
        """
        return self.decoder(
            self.encode(source, source_mask), source_mask, target, target_mask)

In [5]:
class Generator(nn.Module):
    """
    Define the Linear + Softmax generation step to produce output probabilities.
    """
    def __init__(self, d_model, vocab_size):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

#### The Encoder and Decoder Stacks

##### The Encoder Network

The encoder is composed of a stack of $N = 6$ identical layers, followed by layer normalization.

![The Encoder Architecture](imgs/encoder.png)

In [6]:
def layer_clone(layer, N):
    """
    Helper function to produce a stack of N identical layers.
    """
    return nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])

In [7]:
class Encoder(nn.Module):
    """
    The Encoder network, which consists of a stack of N = 6 layers (per the original paper),
    followed by a normalization layer (layer normalization).
    """
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = layer_clone(layer, N)
        self.norm = nn.LayerNorm(layer.size)
    
    def forward(self, x, mask):
        """
        Forward pass through the encoder stack of the input sequence x, with the given mask.
        """
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.norm(x)

Here we implement a custom version of Layer Normalization,

In [8]:
class LayerNorm(nn.Module):
    """
    From scratch implementation of layer normalization.
    """
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps
    
    def forward(self, x):
        """
        Forward pass to normalize the input x.
        """
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

We also incorporate a residual connection around each of the two sublayers before the layer normalization, and thus having the output of each sublayer,
$$\mathtt{LayerNorm} (x + \mathtt{SubLayer}(x)),$$
we also make all sublayers to produce outputs of dimension $d_\texttt{model} = 512$ to facilitate the residual connections as well as the embedding layers.

In [9]:
class SubLayerResidual(nn.Module):
    """
    A residual connection followed by a layer normalization.
    Note for code simplicity the norm is first as opposed to last.
    """
    def __init__(self, size, dropout):
        super(SubLayerResidual, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer):
        """
        Apply residual connection to any sublayer with the same size.
        """
        return x + self.dropout(sublayer(self.norm(x)))

Each layer of the encoder has two sublayers, the first layer is a Multi-Head Self-Attention, and the second is a simple, positionwise fully connected feed forward network.

In [11]:
class EncoderLayer(nn.Module):
    """
    The encoder layer of the Transformer model, consisting of multi-head self-attention
    and a position-wise feed-forward network, with residual connections and layer normalization.
    """
    def __init__(self, d_model, multi_head_self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.multi_head_self_attn = multi_head_self_attn
        self.feed_forward = feed_forward
        self.sublayer = layer_clone(SubLayerResidual(d_model, dropout), 2)
        self.d_model = d_model
    
    def forward(self, x, mask):
        """
        Forward pass through the encoder layer with self-attention and feed-forward network.
        """
        
        # Apply multi-head self-attention sublayer
        x = self.sublayer[0](x, lambda x: self.multi_head_self_attn(x, x, x, mask))
        
        # Apply position-wise feed-forward sublayer and return
        return self.sublayer[1](x, self.feed_forward)